# ENTREGA FINAL

Blanca Gallardo, Ferran Tubert i Agustí Costabella - 15 de febrero de 2026

## Carga de librerías y datos

In [30]:
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
from h2o.display import display
from scipy.stats import f_oneway
import h2o
from h2o.grid.grid_search import H2OGridSearch
from h2o.estimators import H2OGradientBoostingEstimator, H2ORandomForestEstimator, H2OGeneralizedLinearEstimator
import h2o
from h2o.automl import H2OAutoML

In [31]:
from src.MovieDatabase import MovieDatabase

mdb = MovieDatabase(uri="mongodb://localhost:27017/", db_name="tmdb_movies_db")

In [32]:
# Extraer todas las colecciones
movies_docs = list(mdb.movies.find({}))
genres_docs = list(mdb.genres.find({}))
production_companies_docs = list(mdb.production_companies.find({}))
production_countries_docs = list(mdb.production_countries.find({}))
language_docs = list(mdb.languages.find({}))

# Convertir a Polars
movies_df = pl.from_dicts(movies_docs).drop("_id")
genres_df = pl.from_dicts(genres_docs).drop("_id")
production_companies_df = pl.from_dicts(production_companies_docs).drop("_id")
production_countries_df = pl.from_dicts(production_countries_docs).drop("_id")
language_df = pl.from_dicts(language_docs).drop("_id")


In [33]:
movies_lazy = movies_df.lazy()
genres_lazy = genres_df.lazy()
companies_lazy = production_companies_df.lazy()
countries_lazy = production_countries_df.lazy()
languages_lazy = language_df.lazy()

In [34]:
# Movies
display(movies_lazy.select(pl.all()).limit(5).collect())

# Genres
display(genres_lazy.select(pl.all()).limit(5).collect())

# Production companies
display(companies_lazy.select(pl.all()).limit(5).collect())

# Production countries
display(countries_lazy.select(pl.all()).limit(5).collect())

# Languages
display(languages_lazy.select(pl.all()).limit(5).collect())

id,title,original_title,release_date,vote_average,vote_count,popularity,runtime,original_language,revenue,overview,tagline,adult,homepage,budget,genre_ids,company_ids,country_ids,language_ids
i64,str,str,str,f64,i64,f64,i64,str,i64,str,str,bool,str,i64,list[i64],list[i64],list[i64],list[i64]
1333643,"""周刊财经1分快3稳定赚钱计划""","""周刊财经1分快3稳定赚钱计划""","""""",0.0,0,0.0,0,"""zh""",0,"""""","""""",false,"""""",0,[2],[6394],[37],[14]
809167,"""BRZRKR""","""BRZRKR""","""""",0.0,0,1.184,0,"""en""",0,"""The story is about an immortal…","""""",false,"""""",0,[14],[6394],[37],[14]
873426,"""Orchid Boys""","""Orchid Boys""","""2017-01-01""",0.0,0,0.6,9,"""en""",0,"""A tenacious lover and his reti…","""""",false,"""""",0,"[9, 3]",[6394],[179],[68]
771043,"""Body Clock""","""腹腹時計 ハラハラトケー""","""1999-01-01""",0.0,0,0.6,97,"""ja""",0,"""In 1975 a middle-aged anarchis…","""""",false,"""""",0,[9],[160824],[108],"[68, 92, 147]"
529323,"""Big Busty Workout 3""","""Big Busty Workout 3""","""2014-06-25""",0.0,0,0.6,116,"""en""",0,"""You'll love the view at this g…","""""",true,"""""",0,[18],[48936],[179],[14]


genre_id,name
i64,str
0,"""Western"""
1,"""War"""
2,""""""
3,"""Romance"""
4,"""Science Fiction"""


company_id,name
i64,str
0,"""Nowhere Sp. z o. o."""
1,"""Ama Movie AIE"""
2,"""Light In The Attic"""
3,"""Redglass Picures"""
4,"""Nexus Factory"""


country_id,name
i64,str
0,"""Cayman Islands"""
1,"""Marshall Islands"""
2,"""Rwanda"""
3,"""Afghanistan"""
4,"""Suriname"""


language_id,name
i64,str
0,"""Mandarin"""
1,"""Hindi"""
2,"""Manx"""
3,"""Tamil"""
4,"""Malagasy"""


## Limpieza y preparación

In [35]:
# Movies
movies_cols = movies_lazy.collect_schema().names()
movies_nulls = movies_lazy.select([
    pl.col(c).is_null().sum().alias(f"{c}_nulls") for c in movies_cols
]).collect()
display(movies_nulls)

# Genres
genres_cols = genres_lazy.collect_schema().names()
genres_nulls = genres_lazy.select([
    pl.col(c).is_null().sum().alias(f"{c}_nulls") for c in genres_cols
]).collect()
display(genres_nulls)

# Production Companies
companies_cols = companies_lazy.collect_schema().names()
companies_nulls = companies_lazy.select([
    pl.col(c).is_null().sum().alias(f"{c}_nulls") for c in companies_cols
]).collect()
display(companies_nulls)

# Production Countries
countries_cols = countries_lazy.collect_schema().names()
countries_nulls = countries_lazy.select([
    pl.col(c).is_null().sum().alias(f"{c}_nulls") for c in countries_cols
]).collect()
display(countries_nulls)

# Languages
languages_cols = languages_lazy.collect_schema().names()
languages_nulls = languages_lazy.select([
    pl.col(c).is_null().sum().alias(f"{c}_nulls") for c in languages_cols
]).collect()
display(languages_nulls)

id_nulls,title_nulls,original_title_nulls,release_date_nulls,vote_average_nulls,vote_count_nulls,popularity_nulls,runtime_nulls,original_language_nulls,revenue_nulls,overview_nulls,tagline_nulls,adult_nulls,homepage_nulls,budget_nulls,genre_ids_nulls,company_ids_nulls,country_ids_nulls,language_ids_nulls
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


genre_id_nulls,name_nulls
u32,u32
0,0


company_id_nulls,name_nulls
u32,u32
0,0


country_id_nulls,name_nulls
u32,u32
0,0


language_id_nulls,name_nulls
u32,u32
0,0


In [36]:
# Movies
movies_duplicates = movies_lazy.select([
    pl.col("id").is_unique().alias("id_unique")
]).collect()
num_movies_duplicates = (movies_duplicates["id_unique"] == False).sum()
print(f"Duplicados en movies: {num_movies_duplicates}")

# Genres
genres_duplicates = genres_lazy.select([
    pl.col("genre_id").is_unique().alias("id_unique")
]).collect()
num_genres_duplicates = (genres_duplicates["id_unique"] == False).sum()
print(f"Duplicados en genres: {num_genres_duplicates}")

# Production Companies
companies_duplicates = companies_lazy.select([
    pl.col("company_id").is_unique().alias("id_unique")
]).collect()
num_companies_duplicates = (companies_duplicates["id_unique"] == False).sum()
print(f"Duplicados en companies: {num_companies_duplicates}")

# Production Countries
countries_duplicates = countries_lazy.select([
    pl.col("country_id").is_unique().alias("id_unique")
]).collect()
num_countries_duplicates = (countries_duplicates["id_unique"] == False).sum()
print(f"Duplicados en countries: {num_countries_duplicates}")

# Languages
languages_duplicates = languages_lazy.select([
    pl.col("language_id").is_unique().alias("id_unique")
]).collect()
num_languages_duplicates = (languages_duplicates["id_unique"] == False).sum()
print(f"Duplicados en languages: {num_languages_duplicates}")

Duplicados en movies: 0
Duplicados en genres: 0
Duplicados en companies: 0
Duplicados en countries: 0
Duplicados en languages: 0


In [38]:
import polars as pl

movies_df_final = (
    movies_lazy
    .with_columns([
        # Netejar i convertir release_date de forma robusta
        pl.col("release_date")
          .str.strip_chars()
          .replace("", None)
          .str.strptime(pl.Date, "%Y-%m-%d", strict=False)
          .alias("release_date"),

        # adult -> is_adult 0/1
        pl.col("adult").cast(pl.Int8).alias("is_adult")
    ])
    .with_columns([
        # Extraer año y mes
        pl.col("release_date").dt.year().alias("release_year"),
        pl.col("release_date").dt.month().alias("release_month")
    ])
    .select([
        "id",
        "vote_average",
        "vote_count",
        "release_year",
        "release_month",
        "revenue",
        "runtime",
        "is_adult",
        "budget",
        "original_language",
        "popularity",
        pl.col("genre_ids").alias("id_genre"),
        pl.col("language_ids").alias("id_languages"),
        pl.col("company_ids").alias("id_production"),
        pl.col("country_ids").alias("id_country")
    ])
)

display(movies_df_final.collect().head(5))

# Guardar a Parquet
parquet_path = "tmdb_clean.parquet"
movies_df_final.collect().write_parquet(
    parquet_path,
    compression="zstd"
)
print(f"Parquet generado en {parquet_path}")

id,vote_average,vote_count,release_year,release_month,revenue,runtime,is_adult,budget,original_language,popularity,id_genre,id_languages,id_production,id_country
i64,f64,i64,i32,i8,i64,i64,i8,i64,str,f64,list[i64],list[i64],list[i64],list[i64]
1333643,0.0,0,null,null,0,0,0,0,"""zh""",0.0,[2],[14],[6394],[37]
809167,0.0,0,null,null,0,0,0,0,"""en""",1.184,[14],[14],[6394],[37]
873426,0.0,0,2017,1,0,9,0,0,"""en""",0.6,"[9, 3]",[68],[6394],[179]
771043,0.0,0,1999,1,0,97,0,0,"""ja""",0.6,[9],"[68, 92, 147]",[160824],[108]
529323,0.0,0,2014,6,0,116,1,0,"""en""",0.6,[18],[14],[48936],[179]


Parquet generado en tmdb_clean.parquet


## EDA

In [39]:
# Movies
display(movies_lazy.collect_schema())

# Genres
display(genres_lazy.collect_schema())

# Companies
display(companies_lazy.collect_schema())

# Countries
display(countries_lazy.collect_schema())

Schema([('id', Int64),
        ('title', String),
        ('original_title', String),
        ('release_date', String),
        ('vote_average', Float64),
        ('vote_count', Int64),
        ('popularity', Float64),
        ('runtime', Int64),
        ('original_language', String),
        ('revenue', Int64),
        ('overview', String),
        ('tagline', String),
        ('adult', Boolean),
        ('homepage', String),
        ('budget', Int64),
        ('genre_ids', List(Int64)),
        ('company_ids', List(Int64)),
        ('country_ids', List(Int64)),
        ('language_ids', List(Int64))])

Schema([('genre_id', Int64), ('name', String)])

Schema([('company_id', Int64), ('name', String)])

Schema([('country_id', Int64), ('name', String)])

## Modelo

In [40]:
# Check what columns are actually available
print("Available columns in train:", train.columns)
print("\nPredictors list:", predictors)

# Find which predictors are missing
missing_cols = [col for col in predictors if col not in train.columns]
if missing_cols:
    print(f"\nMissing columns: {missing_cols}")


Available columns in train: ['id', 'vote_average', 'vote_count', 'release_year', 'release_month', 'revenue', 'runtime', 'is_adult', 'budget', 'original_language', 'popularity', 'id_genre', 'id_languages', 'id_production', 'id_country']

Predictors list: ['id_genre', 'original_language', 'vote_count', 'budget']


In [41]:
h2o.init()

# Convertir a H2OFrame
df_h2o = h2o.H2OFrame(movies_df_final.collect().to_pandas())

# Definir target y predictores
target = "revenue"
predictors = ["id_genre", "original_language", "vote_count", "budget"]

# Dividir en train y test
train, test = df_h2o.split_frame(ratios=[0.8], seed=42)

# Entrenar AutoML
aml = H2OAutoML(max_models=10, seed=42, project_name="vote_prediction")
aml.train(x=predictors, y=target, training_frame=train)

# Mostrar leaderboard
lb = aml.leaderboard
display(lb.head(rows=lb.nrows))

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,2 hours 0 mins
H2O_cluster_timezone:,Europe/Madrid
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,2 months and 22 days
H2O_cluster_name:,H2O_from_python_ferran_f69hn1
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.264 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
23:19:27.580: AutoML: XGBoost is not available; skipping it.
23:48:46.643: New models will be added to existing leaderboard vote_prediction@@revenue (leaderboard frame=null) with already 12 models.
23:48:46.643: AutoML: XGBoost is not available; skipping it.

██████████████████████████████████████████████████ (cancelled)


H2OJobCancelled: Job<$03017f00000132d4ffffffff$_bf5ff80477c92442a9ce647ca91f4873> was cancelled by the user.

In [ ]:
best_model = aml.leader
perf = best_model.model_performance(test)

print("RMSE:", perf.rmse())
print("MAE:", perf.mae())
print("R2:", perf.r2())
print(aml.leaderboard)

## Optimización

In [ ]:
# ==========================================================
# OPTIMITZACIÓ DEL MILLOR MODEL BASE (GBM)
# ==========================================================

from h2o.estimators import H2OGradientBoostingEstimator
from h2o.grid.grid_search import H2OGridSearch

# Espai d'hiperparàmetres ampliat
gbm_params = {
    "learn_rate": [0.01, 0.05],
    "max_depth": [5, 7],
    "sample_rate": [0.8, 1.0]
}

# Model base coherent amb AutoML
gbm = H2OGradientBoostingEstimator(
    ntrees=700,
    seed=42,
    nfolds=5,
    stopping_rounds=5,
    stopping_metric="RMSE",
    stopping_tolerance=0.001
)

# Grid Search
grid = H2OGridSearch(
    model=gbm,
    grid_id="gbm_manual_optimization",
    hyper_params=gbm_params
)

grid.train(
    x=predictors,
    y=target,
    training_frame=train
)

# Ordenar per RMSE
grid_perf = grid.get_grid(sort_by="rmse", decreasing=False)

print("===== RESULTATS GRID =====")
print(grid_perf)

# Millor model
best_gbm = grid_perf.models[0]

# Avaluació en test
perf_test = best_gbm.model_performance(test)

print("\n===== RESULTATS TEST =====")
print("RMSE:", perf_test.rmse())
print("MAE:", perf_test.mae())
print("R2:", perf_test.r2())

## Evaluación

In [ ]:
# Evaluación del mejor modelo
perf = aml.leader.model_performance(test)
print(perf)